In [1]:
from google import genai
from dotenv import load_dotenv
import os


load_dotenv()

True

In [2]:
google_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])  # Replace with your


def get_answers_from_google_api(
    prompt: str, model: str, config: dict | None = None
) -> str:
    """
    Function to get answers from Google API using the provided prompt.

    Args:
        prompt (str): The input prompt for which the answer is to be generated.
        model (str): The model to use for generating the answer.
        config (dict | None): The configuration for generating the answer.

    Returns:
        str: The generated answer from the Google API.
    """
    response = google_client.models.generate_content(
        model=model, contents=prompt, config=config if config else None
    )

    if response:
        return response
    else:
        return "No response received from the Google API."

# Cost Counter for Model

In [3]:
def cost_count(input_tokens: int, output_tokens: int, model: str) -> float:
    """
    Function to calculate the cost of the API call based on the input and output tokens and the model used.

    Args:
        input_token (int): The number of input tokens.
        output_token (int): The number of output tokens.
        model (str): The model used for the API call.

    Returns:
        float: The total cost of the API call.
    """

    model_costs = {
        "gemini-3.5-flash-lite": {"input": 0.0000015, "output": 0.000009},  # real price
        "gemini-3.5-flash": {"input": 0.0002, "output": 0.0002},  # demo price
        "gemini-4": {"input": 0.0003, "output": 0.0003},  # demo price
    }

    # Get the cost per token for the specified model
    cost_per_token = model_costs.get(model, {"input": 0, "output": 0})

    # Calculate total cost
    total_cost = (input_tokens * cost_per_token["input"]) + (
        output_tokens * cost_per_token["output"]
    )

    return total_cost

In [4]:
estimate_1 = cost_count(
    input_tokens=100, output_tokens=200, model="gemini-3.5-flash-lite"
)
print(estimate_1)

0.00195


In [5]:
estimate_2 = cost_count(input_tokens=100, output_tokens=200, model="gemini-4")
print(estimate_2)

0.09


# Check hallucination

In [32]:
prompts = [
    "Who won the ICC Cricket World Cup in 2038?",
    "What is Chapter 17 of my private diary?",
    "Give me the serial number of Albert Einstein's laptop.",
    "What is the employee ID of John from my company?",
    "Quote page number 257 from this book.",
]

In [33]:
with open("../../mini-project/results/hallutination_result.txt", "a") as f:
    for prompt in prompts:
        response = get_answers_from_google_api(
            prompt=prompt, model="gemini-3.5-flash-lite"
        )
        f.write(f"Prompt : {prompt}\nResponse: {response.text}\n\n\n")
        print(f"Prompt : {prompt}\nResponse: {response.text}\n\n\n")

Prompt : Who won the ICC Cricket World Cup in 2038?
Response: I cannot answer this question as the year 2038 is in the future, and the ICC Cricket World Cup for that year has not yet taken place.



Prompt : What is Chapter 17 of my private diary?
Response: I don't have access to your private diary, as I don't know who you are and I am not connected to your personal devices or accounts. You are the only one who knows what's in Chapter 17!



Prompt : Give me the serial number of Albert Einstein's laptop.
Response: Albert Einstein did not have a laptop. Laptops were invented decades after his death in 1955.



Prompt : What is the employee ID of John from my company?
Response: I do not have access to your company's internal systems, database, or employee records. Therefore, I cannot provide the employee ID for John. You will need to check your company's directory or HR system to find this information.



Prompt : Quote page number 257 from this book.
Response: It looks like you forgot t

# Sampling parameter Comparison

In [36]:
prompt = "Write a 5 ways that ai help in daily life"

In [37]:
top_p_values = [0.0, 0.2, 0.5, 0.8, 1.0]
temprature_values = [0.0, 0.1, 0.5, 0.7, 1.0, 1.2]

combinations = []
for p in top_p_values:
    for temp in temprature_values:
        combinations.append({"top_p": p, "temperature": temp})

In [44]:
model = "gemini-3.5-flash-lite"

In [ ]:
with open("../../mini-project/results/sampling_result.txt", "a") as f:
    for comb in combinations:
        response = get_answers_from_google_api(prompt, model, config=comb)
        block = "=" * 50
        f.write(f"Combination :- {comb} \n Answer : {response.text} \n\n{block}\n\n")